# 05 · 실제 Ollama 응답과 검토 보고서

**핵심 실행 10분** · 동일 모델의 실제 A/B 응답을 보고, 개별 답변 품질과 사용자 실험 효과를 구분합니다.

준비: 저장소 터미널에서 `python -m pip install -r requirements/week12-online.txt` 실행.
VS Code 오른쪽 위 **Select Kernel → 저장소 .venv의 Python 3.11**을 선택합니다.
다른 Notebook의 변수나 실행 순서를 요구하지 않습니다. 이 파일의 셀은 위에서 아래로 실행하세요.

**출처 구분:** CSV는 교수 작성 합성 사용자 데이터입니다. FastAPI 요청은 실제 실습 서비스 실행이며 실제 고객 실험이 아닙니다.
05번의 Ollama만 실제 로컬 모델을 호출합니다. Docker·클라우드 계정·유료 API는 필수가 아닙니다.

## 코드 셀 실행 지도

| 코드 셀 | 실행 | 확인할 결과 |
|---:|---|---|
| 1 | 환경 준비 | Python 3.11과 설정 버전 |
| 2 | 실제 모델 호출 선택 | 실제 모델 호출, qwen2.5:1.5b-instruct, temperature=0·num_predict=128·think=false. |
| 3 | 같은 질문의 실제 A/B 응답 | A/B 실제 답변 2개. 모델 연결 실패 시 '실제 호출 미완료', 대체 답변 없음. |
| 4 | 11주차 품질 기준으로 재검토 | 기존 실행에서는 A/B의 문자열 3개 모두 True, A stop / B length. 자신의 출력은 달라도 됨. |
| 5 | 실제 실행 증거 저장 | 실제 응답이 있으면 reports/my-ollama-responses.json 저장. 미실행이면 저장 생략. |
| 6 | 합성 사례 보고서 재생성 | healthy REVIEW / slow HOLD / srm INVALID, reports/report.html 생성. |
| 7 | 다음 변경의 실행 순서 | 09 API→10 운영→11 품질→12 실험의 다음 확인 표. |

**이번 파일의 역할:** 실제 모델 후보의 준비 상태와 사용자 효과 증거의 차이를 확인하는 짧은 연결 실습입니다. 생성·평가 도구 설치를 다시 배우는 핵심 구간으로 늘리지 않습니다.

**재실행 방법:** Ollama와 모델을 먼저 준비합니다. 셀 3 재실행은 live를 비우고 A/B를 다시 호출합니다. 생성 문구·지연·종료 이유는 환경에 따라 달라질 수 있습니다. 부분 성공이면 두 응답이 모두 있는지 확인한 뒤 셀 4–5로 갑니다.

**코드 셀 1 설명:** 실험을 비교하기 전에 실행 환경과 조건부터 맞춥니다. 파일별 커널은 독립적이므로 각 Notebook의 첫 코드 셀부터 실행합니다.

번호는 코드 셀 순서이며 실행 횟수 `[1]`, `[2]`와 다릅니다. [셀별 교수 설명](https://smilesjcha.github.io/llmops-16week/week12/lecture/instructor/#book-05)

In [ ]:
from pathlib import Path
import sys, json, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week12/lab/analysis.py").exists()), None)
if ROOT is None:
    raise RuntimeError("저장소 폴더를 VS Code로 열고 실행하세요.")
if sys.version_info[:2] != (3, 11):
    raise RuntimeError("저장소 .venv의 Python 3.11 커널을 선택하세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, HTML
from week12.lab.experiment import DEFAULT, assign
from week12.lab.analysis import analyze
from week12.lab.simulate import load_units
print("준비 완료:", sys.version.split()[0], "실험 설정", DEFAULT.version)


## 코드 셀 2 · 실제 모델 호출 선택

Ollama는 11주차의 작은 Instruct 모델을 재사용합니다. API 키나 Docker가 필요 없습니다. 모델이 없으면 `ollama pull qwen2.5:1.5b-instruct`를 먼저 실행하세요.

In [ ]:
RUN_OLLAMA = os.getenv("W12_RUN_OLLAMA", "1") == "1"
print("실제 모델 호출" if RUN_OLLAMA else "명시적 건너뛰기: 실제 LLM 생성 없음")
from week12.lab.ollama import generate, MODEL
print("모델:",MODEL,"/ temperature 0 / num_predict 128 / think false")

**확인과 해석:** 실제 모델 호출, qwen2.5:1.5b-instruct, temperature=0·num_predict=128·think=false.

**화면에서 볼 곳:** RUN_OLLAMA와 MODEL

**설명 핵심:** 11주차의 빠른 모델을 재사용합니다. 실행이 어렵다면 이 셀에서 RUN_OLLAMA=False로 설정하고 다음 셀은 SKIP으로 기록합니다. 호출하지 않은 것을 성공 응답으로 표시하지 않습니다.

**직접 설명:** 빠른 instruct 모델을 써도 어떤 내용 검토가 필요할까요?

## 코드 셀 3 · 같은 질문의 실제 A/B 응답

두 번만 생성합니다. 지연에는 첫 모델 로딩도 포함될 수 있어 버전별 속도 벤치마크로 쓰지 않습니다.

In [ ]:
live = []
if RUN_OLLAMA:
    try:
        for variant in ["A","B"]:
            result = generate("exchange", variant)
            live.append({"variant":variant, **result})
            print(variant,result["answer"],"\n")
    except (RuntimeError,ValueError) as exc:
        print("실제 호출 미완료:",exc)
        print("가상 응답으로 대체하지 않습니다. Ollama 상태를 확인한 후 셀 재실행.")
else:
    print("SKIP: 실제 모델 미실행")

**확인과 해석:** A/B 실제 답변 2개. 모델 연결 실패 시 '실제 호출 미완료', 대체 답변 없음.

**화면에서 볼 곳:** live의 A/B 두 행과 실제 답변 원문

**설명 핵심:** 11주차처럼 같은 질문의 두 답변을 본 것만으로는 사용자 A/B 실험이 아닙니다. 여기서는 후보의 품질을 확인하며, 실제 사용자 배정·행동 관찰이 별도로 필요합니다. 첫 호출 로딩 때문에 두 번의 지연으로 속도 우열도 정하지 않습니다.

**직접 설명:** 두 답변 중 마음에 드는 것을 고르는 것과 사용자 A/B 실험은 왜 다른가요?

## 코드 셀 4 · 11주차 품질 기준으로 재검토

기간·배송비·재고 조건을 사람이 확인합니다. 포함 여부 검사는 의미 정확성을 보장하지 않습니다.

In [ ]:
if live:
    checks = [{"variant":r["variant"],"7일 문자열":"7" in r["answer"],"6000원 문자열":"6000" in r["answer"].replace(",",""),"재고 문자열":"재고" in r["answer"],"종료 이유":r["done_reason"],"실제 응답":r["answer"]} for r in live]
    display(pd.DataFrame(checks))
    print("문자열 검사와 사람의 의미 검토를 구분합니다. done_reason=length는 잘림 가능성.")
else:
    print("실제 생성 결과 없음: 표를 만들지 않습니다.")

**확인과 해석:** 기존 실행에서는 A/B의 문자열 3개 모두 True, A stop / B length. 자신의 출력은 달라도 됨.

**화면에서 볼 곳:** 문자열 열보다 실제 응답과 done_reason

**설명 핵심:** 단어가 들어 있어도 의미 오류·반복·문장 잘림이 남을 수 있습니다. length는 출력 상한에 도달한 신호입니다. 실제 B 후보의 문제를 합성 데이터의 좋은 성과로 덮지 않습니다.

**직접 설명:** 조건이 빠진 B를 온라인 실험에 바로 넣으면 어떤 위험이 있나요?

## 코드 셀 5 · 실제 실행 증거 저장

모델 응답과 합성 사용자 실험 결과는 다른 파일에 보관합니다. 사용자의 답변 만족도를 생성 결과에서 만들어내지 않습니다.

In [ ]:
out = ROOT/"week12/lab/reports"
out.mkdir(exist_ok=True,parents=True)
if live:
    (out/"my-ollama-responses.json").write_text(json.dumps({"source":"actual_local_ollama","responses":live},ensure_ascii=False,indent=2))
    print(out/"my-ollama-responses.json")
else:
    print("실제 응답 파일 저장 생략")

**확인과 해석:** 실제 응답이 있으면 reports/my-ollama-responses.json 저장. 미실행이면 저장 생략.

**화면에서 볼 곳:** source=actual_local_ollama와 실제 응답 수

**설명 핵심:** 이 파일은 모델 실행 증거입니다. 사용자의 만족도나 해결 결과를 측정한 파일은 아닙니다. 일부만 성공했다면 부분 실행이라고 설명합니다.

**직접 설명:** 이 파일만으로 ‘B가 해결률을 9%p 높였다’고 말할 수 있나요?

## 코드 셀 6 · 합성 사례 보고서 재생성

이 보고서는 교수 작성 2400명 스냅샷의 계산 결과입니다. 방금 생성한 실제 응답과 합치지 않습니다.

In [ ]:
from week12.lab.report import build_reports
reports = build_reports(out)
print({k:v["report"]["decision"] for k,v in reports.items()})
print("브라우저에서 열 파일:",out/"report.html")

**확인과 해석:** healthy REVIEW / slow HOLD / srm INVALID, reports/report.html 생성.

**화면에서 볼 곳:** 보고서에서 slow 선택→사용자 수→차이·구간→p95→판단 이유

**설명 핵심:** 보고서는 합성 사용자 CSV를 계산한 결과입니다. 방금 생성한 응답이 9%p 개선을 만들었다고 말하지 않습니다. 화면의 사례 선택기를 바꾸며 보류와 데이터 무효를 구분합니다.

**직접 설명:** 왜 실습 서비스 화면과 종료된 합성 실험 보고서를 따로 제공할까요?

## 코드 셀 7 · 다음 변경의 실행 순서

출력 길이 축소는 속도를 개선할 수도 있지만 근거 누락을 만들 수 있습니다. 새 설정은 기존 실험 기록과 분리합니다.

In [ ]:
next_plan = pd.DataFrame([
    ["09 API","request_id·응답 계약 유지"],
    ["10 운영","오류·지연·중단 상태 기록"],
    ["11 품질","기간·비용·재고 조건 회귀 평가"],
    ["12 실험","새 버전·새 ID로 배정·사용자 결과 비교"],
],columns=["단원","다음 확인"])
display(next_plan)

**확인과 해석:** 09 API→10 운영→11 품질→12 실험의 다음 확인 표.

**화면에서 볼 곳:** 각 주차가 남기는 서로 다른 증거

**설명 핵심:** 서비스 계약, 운영 상태, 답변 품질, 사용자 효과는 같은 서비스의 서로 다른 질문입니다. 마지막에는 다음 변경 하나와 새 실험 전에 고정할 기준을 학생 자신의 말로 설명하게 합니다.

**직접 설명:** 다음 실험을 시작하기 전에 문서에 먼저 고정할 것은 무엇인가요?

## 복습과 다음 단계

과제에서 실제 Ollama 호출은 선택입니다. 필수는 API 기록 한 장과 합성 slow 사례의 판단 한 페이지입니다. 모델이 느리면 01–04 핵심 경로를 먼저 완료하고 05를 복습하세요.

실행 오류는 Kernel을 다시 시작한 뒤 위에서부터 실행해 확인합니다. `ModuleNotFoundError`는 설치 터미널과 Notebook 커널이 같은 `.venv`인지 확인하세요.
실험 결과는 `week12/lab/reports/`에 저장하고 공개 GitHub에 학생 답안·개인정보를 올리지 않습니다.
[실행 안내](../WALKTHROUGH.md) · [100분 강의안](https://smilesjcha.github.io/llmops-16week/week12/lecture/) · [9–12주차 과제](https://smilesjcha.github.io/llmops-16week/week12/assignment/)
